# Wahla — Financial Digital Twin (backend demo)

> A dashboard tells you what you spent; a chatbot tells you what your data says;
> **our Twin tells you who you will be in 24 months if you make this decision** —
> and shows exactly which parts of your financial self change and why.

This notebook runs the **entire backend** — data cleaning, feature engineering,
the Twin engine, memory/timeline, simulation, and the live FastAPI service — end
to end on Google Colab. It clones the public repo, so it always runs the latest
pushed code. The React frontend isn't included here; it's a browser/device UI,
not something Colab can meaningfully render — everything it talks to (the API)
is fully live in this notebook.

Repo: https://github.com/AbdullahAlbadri/Wahla


## 0. Get the code

In [ ]:
import os

if not os.path.exists("Wahla"):
    !git clone --quiet https://github.com/AbdullahAlbadri/Wahla.git

%cd Wahla
!git pull --quiet


In [ ]:
!pip install -q fastapi uvicorn pydantic pandas nest-asyncio requests


## 1. Build the twins

Raw Berka export (1,056,320 real transactions, 4,500 clients) →
`twin/data_loader.py` (cleaning) → `twin/features.py` (behavioral features) →
`twin/engine.py` (persistent Twin state) → `twin/memory.py` (timeline).

This single run also demonstrates two judge-mode views for account 19:
- `--account 19` — demo simulations (buy a car, compare car-loan vs cash vs invest)
- `--inspect 19` — full data lineage: raw rows → features → state → event → new state → why

Takes a few minutes on Colab's CPU (builds all 4,500 twins to match the accounts
the API expects). Pass `--limit 500` instead if you just want a fast smoke test.


In [ ]:
!python3 build_twins.py --account 19 --inspect 19


## 2. Architecture isolation proof

Statically verifies (via AST inspection, not convention) that nothing downstream
of the Twin — `engine.py`, `simulation.py`, `diff.py`, `explain.py`,
`personality.py`, `report.py`, `validation.py` — ever imports `data_loader`,
`pandas`, or reads a raw file. Once a Twin is built, everything else operates on
state alone.


In [ ]:
!python3 check_architecture.py


## 3. Live cleaning boundary — the connect-time equivalent of the batch pipeline

`load_live_transactions()` / `load_live_loans()` in `twin/data_loader.py` are what
a real Open Banking connection lands on before anything reaches feature
engineering — same typing/dedup/dropna contract as the Berka batch loader, but
built for a live JSON feed (mixed date formats, occasional missing fields,
duplicate records — all of which get dropped or normalized here, not later).

The sample below deliberately includes: a duplicate transaction, a transaction
with an unparseable date, one with a missing amount, and a loan missing its
`loan_id`. Watch the row count drop from what goes in to what survives.


In [ ]:
import sys
sys.path.insert(0, ".")
from twin.data_loader import load_live_transactions, load_live_loans

dirty_transactions = [
    {"trans_id": "a1", "account_id": 99001, "trans_date": "2024-02-01T09:00:00",
     "amount": 6000, "balance": 6000, "trans_type": "credit", "category": "salary"},
    {"trans_id": "a2", "account_id": 99001, "trans_date": "2024-02-05",
     "amount": -800, "balance": 5200, "trans_type": "debit", "category": "household"},
    {"trans_id": "a2", "account_id": 99001, "trans_date": "2024-02-05",
     "amount": -800, "balance": 5200, "trans_type": "debit", "category": "household"},
    {"trans_id": "a3", "account_id": 99001, "trans_date": "not-a-date",
     "amount": 50, "trans_type": "debit"},
    {"trans_id": "a4", "account_id": 99001, "trans_date": "2024-02-10",
     "amount": None, "trans_type": "debit"},
]
tx = load_live_transactions(dirty_transactions)
print(f"transactions: {len(dirty_transactions)} in -> {len(tx)} survived cleaning")
print(tx[["trans_id", "trans_date", "amount", "trans_type", "category"]])

dirty_loans = [
    {"loan_id": 501, "account_id": 99001, "granted_date": "2023-05-01",
     "amount": 15000, "duration": 24, "payments": 700, "status": "running_ok"},
    {"loan_id": None, "account_id": 99001, "granted_date": "2023-06-01",
     "amount": 5000, "duration": 6, "payments": 900, "status": "running_ok"},
]
loans = load_live_loans(dirty_loans)
print(f"\nloans: {len(dirty_loans)} in -> {len(loans)} survived cleaning")
print(loans[["loan_id", "account_id", "status"]])


## 4. Run the live API inside this notebook

`api.py` is a normal FastAPI app — this launches it on `127.0.0.1:8000` in a
background thread so later cells in this same notebook can call it exactly like
the real frontend would.


In [ ]:
import threading, time
import nest_asyncio
import uvicorn

nest_asyncio.apply()

import importlib
import api as wahla_api
importlib.reload(wahla_api)

def _run_server():
    uvicorn.run(wahla_api.app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=_run_server, daemon=True).start()
time.sleep(2)

import requests
BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/api/accounts").json())


### 4a. Connect a live account — the full HTTP path through the cleaning layer

Same dirty sample as section 3, this time sent over HTTP to `POST /api/connect`,
proving the cleaning happens before the Twin is built, not just when called
directly from Python.


In [ ]:
connect_payload = {
    "account_id": 99001,
    "transactions": dirty_transactions,
    "loans": dirty_loans,
}
r = requests.post(f"{BASE}/api/connect", json=connect_payload)
print("status:", r.status_code)
twin_state = r.json()
print("financial_health_score:", twin_state.get("financial_health_score"))
print("current_balance:", twin_state.get("current_balance"))


### 4b. Simulate a decision on a real demo account

In [ ]:
decision = {"type": "loan", "monthly": 900, "months": 24, "hasDownPayment": False}
r = requests.post(f"{BASE}/api/simulate/19", json=decision)
print(r.status_code)
r.json()


In [ ]:
r = requests.get(f"{BASE}/api/alternatives/19", params={"monthly": 900, "months": 24})
print(r.status_code)
r.json()


## Where to read more

- `README.md` — architecture overview and pipeline diagram
- `ARCHITECTURE.md` — full design rationale
- `HACKATHON.md` — the 90-second "why this is a real Digital Twin, not a dashboard" answer
- `twin/` — engine, features, simulation, memory, personality, diff, explain,
  report, validation (everything downstream of the Twin, isolated from raw data)
